# Data loader for both libraries (Pytorch and Tensorflow)

In [ ]:
import sys

common_modules_path = "/content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50"
sys.path.append(common_modules_path)
print(f"Updated sys.path: {sys.path}")

Updated sys.path: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50']


In [3]:
import config

## Base Data Loader

In [5]:
from base_loader import BaseDataLoader

In [ ]:
# Test 1: Basic loading
training_dl = BaseDataLoader(config.COVIDQU_PATH, split="Train", is_train_shuffle=True, seed=config.SEED)
print(f"Train samples: {len(training_dl)}")
print(f"Class mapping: {training_dl.get_class_mapping()}")

Train samples: 3728
Class mapping: {'COVID-19': 0, 'Non-COVID': 1, 'Normal': 2}


In [10]:
# Test 2: Access individual samples
img_path, label = training_dl[0]
print(f"First sample: {img_path} -> label {label}")

First sample: /content/drive/MyDrive/ai-datasets/covidqu/Train/COVID-19/covid_4020.png -> label 0


## PyTorch Wrapper

In [16]:
import torch
import torchvision.transforms as T
from torch_dataset import TorchMedicalDataset

# T.RandomHorizontalFlip() is a data augmentation technique that Randomly flips images horizontally with a 50% probability.
# Expands the training dataset without collecting new images, Prevents overfitting by exposing the model to mirrored versions
# Helps the model learn orientation-invariant features
# TODO: Ensure both use the same probability (default 0.5).

torch_train_transform = T.Compose(
    [
        T.Resize(config.IMG_SIZE), # resize to match ResNet-50 expected input image size
        T.RandomHorizontalFlip(), # data augmentation
        T.ToTensor(), # Scale to [0,1]
        T.Normalize(config.MEAN, config.STD) # normalize the values
    ]
)

torch_dataset = TorchMedicalDataset(training_dl, torch_train_transform)

In [17]:
from torch.utils.data import DataLoader

# initialize torch DataLoader library
torch_dataloader = DataLoader(
    torch_dataset,
    batch_size=config.BATCH_SIZE, # depend upon memory size, Large for stable gradients, small for generalization
    shuffle=False, # already shuffled in training_dl
)

In [18]:
images, labels = next(iter(torch_dataloader))
print(f"Batch images shape: {images.shape}")  # Expected: [B, C, H, W]
print(f"Batch labels: {labels}")
print(f"Unique classes: {torch.unique(labels)}")

Batch images shape: torch.Size([32, 3, 224, 224])
Batch labels: tensor([0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 2, 0, 0, 2, 2, 0, 1, 1, 1, 2, 0, 2, 1,
        0, 0, 0, 1, 0, 2, 0, 0])
Unique classes: tensor([0, 1, 2])


## TensorFlow Wrapper

In [11]:
from tf_dataset import create_tf_dataset

# Create TF dataset (no augmentation for validation; use augment=True for training)
tf_train_dataset = create_tf_dataset(training_dl, config.IMG_SIZE, config.MEAN, config.STD, batch_size=config.BATCH_SIZE, augment=True)

# Test one batch
for images, labels in tf_train_dataset.take(1):
    print(f"Images shape: {images.shape}")  # Expected: (batch, 224, 224, 3)
    print(f"Labels: {labels.numpy()}")
    print(f"Image value range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")

Images shape: (32, 224, 224, 3)
Labels: [0 1 0 0 0 1 0 0 1 0 0 2 0 0 2 2 0 1 1 1 2 0 2 1 0 0 0 1 0 2 0 0]
Image value range: [-2.12, 2.64]


## Validate Consistancy Across both Library Data Wrapers

In [ ]:
# spotcheck_consistency.py

# 1. Same base loader for both
base_loader = BaseDataLoader(config.COVIDQU_PATH, split="Train", is_train_shuffle=True, seed=config.SEED)

# 2. PyTorch transforms
transform = T.Compose([
    T.Resize(config.IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=config.MEAN, std=config.STD)
])

# 3. PyTorch dataset
pt_dataset = TorchMedicalDataset(base_loader, transform=transform)
pt_loader = DataLoader(pt_dataset, batch_size=1, shuffle=False)

# 4. TensorFlow dataset (no augmentation for comparison)
tf_dataset = create_tf_dataset(base_loader, config.IMG_SIZE, config.MEAN, config.STD, batch_size=1, augment=False)

In [ ]:
# 5. Compare first 5 samples
print("Sample consistency check:")
for i in range(5):
    # Get PyTorch sample
    pt_img, pt_label = next(iter(pt_loader))

    # Get TensorFlow sample
    tf_img, tf_label = next(iter(tf_dataset))

    # Convert TF tensor to numpy for comparison
    tf_img_np = tf_img.numpy().squeeze()  # Remove batch dimension
    tf_img_np = np.transpose(tf_img_np, (2, 0, 1))  # HWC -> CHW for comparison

    pt_img_np = pt_img.squeeze().numpy()

    print(f"Sample {i+1}:")
    print(f"  Label match: {pt_label.item() == tf_label.numpy()[0]}")
    print(f"  Image shape PT: {pt_img_np.shape}, TF: {tf_img_np.shape}")
    print(f"  Mean pixel diff: {np.abs(pt_img_np - tf_img_np).mean():.6f}")
    print()

Sample consistency check:
Sample 1:
  Label match: True
  Image shape PT: (3, 224, 224), TF: (3, 224, 224)
  Mean pixel diff: 0.005749

Sample 2:
  Label match: True
  Image shape PT: (3, 224, 224), TF: (3, 224, 224)
  Mean pixel diff: 0.005749

Sample 3:
  Label match: True
  Image shape PT: (3, 224, 224), TF: (3, 224, 224)
  Mean pixel diff: 0.005749

Sample 4:
  Label match: True
  Image shape PT: (3, 224, 224), TF: (3, 224, 224)
  Mean pixel diff: 0.005749

Sample 5:
  Label match: True
  Image shape PT: (3, 224, 224), TF: (3, 224, 224)
  Mean pixel diff: 0.005749



## Result Conclusion

Results are consistent and excellent — identical labels, shapes, and near-zero mean pixel difference (0.0057, essentially floating-point precision variance) confirm both frameworks load and preprocess data identically.